# PJM Load Forecasting by Hour-of-Day Experts

This notebook creates 24 separate datasets by hour of day, trains one version of each model per hour, merges the timestamped predictions back together, and then evaluates the merged model families.

## Setup

The notebook reuses the helper functions from `experiments/load_forecasting_by_hour.py` so the notebook and script stay synchronized.

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt

ROOT = Path.cwd().resolve().parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from experiments.load_forecasting_by_hour import (
    HOUR_QRC_TEST,
    HOUR_QRC_TRAIN,
    HOUR_WASHOUT,
    HOURS,
    OUT_DIR,
    fit_full_hourly_classical,
    fit_hourly_experts,
    prepare_hourly_dataframe,
    print_table,
    save_hourly_plots,
)

plt.style.use('seaborn-v0_8-whitegrid')
len(HOURS)

## Build the 24 Hourly Datasets

In [ ]:
train, test, feature_cols = prepare_hourly_dataframe()
print(f'Train rows: {len(train):,}')
print(f'Test rows:  {len(test):,}')
print(f'Hour buckets: {len(HOURS)}')
print(f'Per-hour QRC window: {HOUR_QRC_TRAIN} train / {HOUR_QRC_TEST} test / washout {HOUR_WASHOUT}')

## Full-Year Classical Experts by Hour

In [ ]:
full_merged, full_results = fit_full_hourly_classical(train, test, feature_cols)
print_table(full_results, 'Classical by-hour experts - full 2018 test set')
full_results

## Matched By-Hour Classical vs QRC Comparison

In [ ]:
merged, sub_results, timings = fit_hourly_experts(train, test, feature_cols)
print_table(sub_results, 'By-hour experts - matched classical vs QRC comparison')
timings

## Inline Plots for the By-Hour Experiment

In [ ]:
save_hourly_plots(merged, sub_results, full_results)

## Notes

- Each hour of day gets its own expert model family.
- Predictions are merged back in chronological order before computing the final metrics.
- The resulting plots are directly comparable to the original single-model-family experiment.